# VINS AI VIDEO STUDIO — Cloud Renderer
Run cells from top to bottom in Google Colab. The free runtime is temporary; download your MP4 before closing it.

In [1]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core
!pip -q install edge-tts pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-noto-mono.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../fonts-noto-mono_20201225-2_all.deb ...
Unpacking fonts-noto-mono (20201225-2) ...
Selecting previously unselected package fonts-noto-core.
Preparing to unpack .../fonts-noto-core_20201225-2_all.deb ...
Unpacking fonts-noto-core (20201225-2) ...
Setting up fonts-noto-mono (20201225-2) ...
Setting up fonts-noto-core (20201225-2) ...
Processing triggers for fontconfig (2.15.0-1.1ubuntu2) ...


In [4]:

import os
import json
import zipfile
import shutil
import subprocess
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter
from IPython.display import display, FileLink
from google.colab import files

# --------------------------------------------------
# 1. UPLOAD AND EXTRACT PROJECT
# --------------------------------------------------

print("Upload VINS_AI_Video_Project.zip")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No project ZIP uploaded.")

zip_name = next(iter(uploaded))

work = Path("/content/vins_work")
out = Path("/content/vins_output")

if work.exists():
    shutil.rmtree(work)

if out.exists():
    shutil.rmtree(out)

work.mkdir(parents=True)
out.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(work)

project_file = work / "project.json"

if not project_file.exists():
    raise FileNotFoundError("project.json is missing from the ZIP.")

project = json.loads(project_file.read_text(encoding="utf-8"))

W, H = map(int, project.get("format", "1080x1920").split("x"))

brand = project.get("brand", "VINS ENTERPRISES")
contact = project.get("contact", "")
scenes = project.get("scenes", [])

if not scenes:
    raise ValueError("No scenes found in project ZIP.")

# --------------------------------------------------
# 2. KANNADA FONT
# --------------------------------------------------

font_candidates = [
    "/usr/share/fonts/truetype/noto/NotoSansKannada-Regular.ttf",
    "/usr/share/fonts/truetype/noto/NotoSansKannadaUI-Regular.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansKannada-Regular.ttf"
]

def font(size):
    for path in font_candidates:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    raise RuntimeError("Kannada font not found. Check Noto font installation.")

# --------------------------------------------------
# 3. PROJECT ASSETS
# --------------------------------------------------

assets = project.get("assets", {})

image_paths = [
    work / p for p in assets.get("images", [])
    if (work / p).exists()
]

logo_rel = assets.get("logo")
logo_path = work / logo_rel if logo_rel and (work / logo_rel).exists() else None

music_rel = assets.get("music")
music_path = work / music_rel if music_rel and (work / music_rel).exists() else None

def cover(img, size):
    img = ImageOps.exif_transpose(img).convert("RGB")
    return ImageOps.fit(
        img, size,
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.5)
    )

def wrap_text(draw, text, fnt, max_width):
    words = text.split()
    lines = []
    line = ""

    for word in words:
        test = (line + " " + word).strip()

        if draw.textbbox((0, 0), test, font=fnt)[2] <= max_width:
            line = test
        else:
            if line:
                lines.append(line)
            line = word

    if line:
        lines.append(line)

    return lines

# --------------------------------------------------
# 4. CREATE SCENE IMAGES
# --------------------------------------------------

slide_files = []

for i, scene in enumerate(scenes):

    bg = Image.new("RGB", (W, H), (13, 20, 42))

    if image_paths:
        try:
            photo = Image.open(image_paths[i % len(image_paths)])
            photo = cover(photo, (W, H))
            photo = photo.filter(ImageFilter.GaussianBlur(1))
            bg = Image.blend(bg, photo, 0.58)
        except Exception as e:
            print("Skipping image:", e)

    draw = ImageDraw.Draw(bg, "RGBA")

    panel_top = int(H * 0.48)

    draw.rectangle(
        (0, panel_top, W, H),
        fill=(5, 10, 25, 205)
    )

    draw.rectangle(
        (0, 0, W, int(H * 0.12)),
        fill=(6, 10, 24, 175)
    )

    title = scene.get("title", f"SCENE {i+1}")

    draw.text(
        (int(W * 0.07), int(H * 0.045)),
        brand,
        font=font(max(22, int(W * 0.035))),
        fill=(255, 255, 255, 255)
    )

    draw.text(
        (int(W * 0.07), int(H * 0.53)),
        title,
        font=font(max(28, int(W * 0.055))),
        fill=(196, 181, 253, 255)
    )

    narration = scene.get("narration", "")

    text_font = font(max(22, int(W * 0.038)))

    lines = wrap_text(draw, narration, text_font, int(W * 0.84))

    y = int(H * 0.62)
    line_h = int(H * 0.052)

    for line in lines[:7]:
        draw.text(
            (int(W * 0.07), y),
            line,
            font=text_font,
            fill=(255, 255, 255, 255),
            stroke_width=1,
            stroke_fill=(0, 0, 0, 180)
        )
        y += line_h

    # Footer
    draw.rectangle(
        (0, int(H * 0.91), W, H),
        fill=(8, 12, 28, 220)
    )

    draw.text(
        (int(W * 0.05), int(H * 0.935)),
        contact[:120],
        font=font(max(14, int(W * 0.022))),
        fill=(255, 255, 255, 255)
    )

    if logo_path:
        try:
            lg = Image.open(logo_path).convert("RGBA")
            lg.thumbnail((int(W * 0.20), int(H * 0.12)))
            bg.paste(
                lg,
                (W - lg.width - int(W * 0.05), int(H * 0.14)),
                lg
            )
        except Exception as e:
            print("Logo skipped:", e)

    slide_path = out / f"slide_{i:03d}.png"
    bg.save(slide_path)
    slide_files.append(slide_path)

print("Created", len(slide_files), "scene images.")

# --------------------------------------------------
# 5. GENERATE KANNADA VOICEOVER
# --------------------------------------------------

script = project.get("script", "").strip()

if not script:
    script = " ".join(
        s.get("narration", "") for s in scenes
    ).strip()

if not script:
    raise ValueError("No narration script found.")

voice = project.get("voice") or "kn-IN-GaganNeural"
audio = out / "voiceover.mp3"

try:
    import edge_tts

    async def make_voice():
        communicate = edge_tts.Communicate(script, voice)
        await communicate.save(str(audio))

    # IMPORTANT: Use await, NOT asyncio.run(), in Google Colab.
    await make_voice()

    if not audio.exists() or audio.stat().st_size == 0:
        raise RuntimeError("Voice service did not create an audio file.")

    print("Kannada voice generated successfully:", voice)

except Exception as e:

    print("Automatic voice generation failed:", str(e))
    print("Please upload your narration audio file.")

    if audio.exists():
        audio.unlink()

    alt = files.upload()

    if not alt:
        raise RuntimeError("Narration audio was not uploaded.")

    fn = next(iter(alt))
    shutil.copy(fn, audio)

# --------------------------------------------------
# 6. CHECK AUDIO DURATION
# --------------------------------------------------

def probe_duration(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    return float(result.stdout.strip())

if not audio.exists():
    raise RuntimeError("No narration audio available.")

total = probe_duration(audio)

if total <= 0:
    raise RuntimeError("Invalid narration audio duration.")

weights = [
    max(1, len(s.get("narration", "").split()))
    for s in scenes
]

sumw = sum(weights)

durations = [
    total * weight / sumw
    for weight in weights
]

print("Narration duration:", round(total, 2), "seconds")

# --------------------------------------------------
# 7. CREATE VIDEO CLIPS WITH SLOW ZOOM
# --------------------------------------------------

clips = []

for i, (slide, dur) in enumerate(zip(slide_files, durations)):

    clip = out / f"clip_{i:03d}.mp4"

    # d=1 prevents the zoompan filter from multiplying
    # the intended output duration.
    vf = (
        f"scale={W}:{H},"
        f"zoompan=z='min(zoom+0.00035,1.08)':"
        f"d=1:s={W}x{H}:fps=25,"
        f"format=yuv420p"
    )

    print(f"Rendering scene {i+1}/{len(slide_files)}")

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loop", "1",
            "-framerate", "25",
            "-i", str(slide),
            "-t", str(dur),
            "-vf", vf,
            "-r", "25",
            "-c:v", "libx264",
            "-preset", "ultrafast",
            "-pix_fmt", "yuv420p",
            str(clip)
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    clips.append(clip)

# --------------------------------------------------
# 8. JOIN ALL SCENES
# --------------------------------------------------

concat = out / "clips.txt"

concat.write_text(
    "".join(f"file '{p.as_posix()}'\n" for p in clips),
    encoding="utf-8"
)

silent = out / "silent.mp4"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(concat),
        "-c", "copy",
        str(silent)
    ],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

# --------------------------------------------------
# 9. CREATE KANNADA SUBTITLES
# --------------------------------------------------

def stamp(sec):
    whole = int(sec)
    ms = int(round((sec - whole) * 1000))

    if ms == 1000:
        whole += 1
        ms = 0

    return (
        f"{whole//3600:02}:"
        f"{(whole%3600)//60:02}:"
        f"{whole%60:02},{ms:03}"
    )

srt = []
cursor = 0.0

for i, (scene, duration) in enumerate(zip(scenes, durations)):

    start = cursor
    end = cursor + duration

    narration = scene.get("narration", "")

    srt.append(
        f"{i+1}\n"
        f"{stamp(start)} --> {stamp(end)}\n"
        f"{narration}\n"
    )

    cursor = end

srt_path = out / "VINS_Subtitles.srt"

srt_path.write_text(
    "\n".join(srt),
    encoding="utf-8"
)

# --------------------------------------------------
# 10. ADD VOICE AND BACKGROUND MUSIC
# --------------------------------------------------

final = out / "VINS_Promo.mp4"

if music_path:

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(silent),
            "-i", str(audio),
            "-stream_loop", "-1",
            "-i", str(music_path),
            "-filter_complex",
            "[1:a]volume=1.0[voice];"
            "[2:a]volume=0.12[music];"
            "[voice][music]amix=inputs=2:duration=first:"
            "dropout_transition=2[a]",
            "-map", "0:v",
            "-map", "[a]",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            str(final)
        ],
        check=True
    )

else:

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(silent),
            "-i", str(audio),
            "-map", "0:v",
            "-map", "1:a",
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest",
            str(final)
        ],
        check=True
    )

# --------------------------------------------------
# 11. DOWNLOAD FINAL VIDEO AND SUBTITLES
# --------------------------------------------------

if not final.exists() or final.stat().st_size == 0:
    raise RuntimeError("Final video was not created.")

print("======================================")
print("VINS VIDEO RENDERING COMPLETE!")
print("Video:", final)
print("Subtitles:", srt_path)
print("======================================")

display(FileLink(str(final)))
display(FileLink(str(srt_path)))

from google.colab import files

files.download('/content/vins_output/VINS_Promo.mp4')


Upload VINS_AI_Video_Project.zip


Saving VINS_AI_Video_Project.zip to VINS_AI_Video_Project (1).zip
Created 6 scene images.
Kannada voice generated successfully: kn-IN-GaganNeural
Narration duration: 31.08 seconds
Rendering scene 1/6
Rendering scene 2/6
Rendering scene 3/6
Rendering scene 4/6
Rendering scene 5/6
Rendering scene 6/6
VINS VIDEO RENDERING COMPLETE!
Video: /content/vins_output/VINS_Promo.mp4
Subtitles: /content/vins_output/VINS_Subtitles.srt


/content/vins_output/VINS_Promo.mp4

/content/vins_output/VINS_Subtitles.srt